# Week 5, Day 4: Judge the answers, rank the sources

**Lesson:** [week5/day4.ipynb](../../../week5/day4.ipynb), *Evaluation*. **About 40 minutes.**

**What you practice here**

- An LLM as the judge, as the lesson's `evaluate_answer` does: a pydantic schema whose field descriptions set the 1-to-5 scores, a prompt that gives the judge the question, the generated answer and the reference answer, and one LiteLLM `completion` call that asks for that schema, read back with `model_validate_json`.
- Scoring retrieval by rank, as the lesson's `evaluate_retrieval` does: the reciprocal rank behind MRR, and nDCG, which scores the whole ranking, with the file that holds the answer as what makes a chunk relevant.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week5/day4", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson scores a RAG system twice. It scores retrieval with MRR, nDCG and keyword coverage, and it scores answers
with an LLM as the judge, which compares each answer with a reference answer and gives accuracy, completeness and
relevance scores from 1 to 5. That code lives in `evaluation/eval.py`, which the notebook calls. Write the same ideas
for your WWE questions: a judge of your own, and the two ranking scores, reciprocal rank and nDCG, where a retrieved
chunk counts as relevant when it comes from the file that holds the answer, as the questions in your plan name it.

The function that calls the judge, `judge`, takes LiteLLM's `completion` function as its first parameter and calls it
itself, as `evaluation/eval.py` does. The checks pass a stand-in shaped like `completion`: it records what you send
and answers with canned JSON, so nothing is spent.

## What each call should give

```python
judge_messages("Who won the first Royal Rumble match?", "Jim Duggan.",
               "Hacksaw Jim Duggan won the first Royal Rumble match, in 1988.")
# [{'role': 'system', 'content': 'You are an expert evaluator...'},
#  {'role': 'user', 'content': 'Question:\n...\n\nGenerated answer:\n...\n\nReference answer:\n...'}]

verdict = judge(completion, "Who won the first Royal Rumble match?", "Jim Duggan.",
                "Hacksaw Jim Duggan won the first Royal Rumble match, in 1988.")
# one call with the model gpt-4.1-nano, the messages from judge_messages, and AnswerEval as the response format
verdict.accuracy, verdict.completeness, verdict.relevance
# (5.0, 3.0, 5.0)    (the judge's scores)

reciprocal_rank(docs, "plan/docs/events/royal-rumble-1988.md")
# 0.5 when the first document from events/royal-rumble-1988.md comes second; 0.0 when none does

ndcg(docs, "plan/docs/events/royal-rumble-1988.md")
# 0.6509 when the documents come from: another file, that file, another file, that file
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import math

from pydantic import BaseModel, Field

### 1 · `class AnswerEval(BaseModel)`

The judge's verdict, as a pydantic model the judge must fill. Give it exactly four fields: `feedback`, a string, and
`accuracy`, `completeness` and `relevance`, three floats. Describe each field to the model in 20 characters or more.
The three score descriptions state the scale and what its ends mean, such as "from 1 (wrong) to 5 (perfectly
accurate)": the check looks for "1" and "5" in each. The judge reads these descriptions when `AnswerEval` is the
format it must answer in, as in the lesson's `evaluation/eval.py`.

<details><summary>Hint</summary>

```python
class AnswerEval(BaseModel):
    feedback: str = Field(description="...")
    accuracy: float = Field(description="... from 1 (...) to 5 (...)")
```

</details>

In [ ]:
class AnswerEval(BaseModel):
    # Write the four fields here, as described above.
    pass

In [ ]:
check("AnswerEval")

### 2 · `judge_messages(question, answer, reference)`

Builds the judge's messages: a system message, then a user message. The system message, of 20 characters or more,
makes the model an expert evaluator that compares a generated answer with a reference answer. The user message holds
the question, the generated answer and the reference answer, each under its own label, and asks for feedback and for
accuracy, completeness and relevance scores from 1 to 5. Return the list of the two messages, each a dict with a
`"role"` and a `"content"`.

<details><summary>Hint</summary>

```python
user = f"Question:\n{question}\n\nGenerated answer:\n{answer}\n\nReference answer:\n{reference}\n\n..."
return [{"role": "system", "content": system}, {"role": "user", "content": user}]
```

</details>

In [ ]:
def judge_messages(question, answer, reference):
    # Write judge_messages here, as described above.
    raise NotImplementedError("judge_messages is not written yet")

In [ ]:
check("judge_messages")

### 3 · `judge(completion, question, answer, reference, model="gpt-4.1-nano")`

Asks the judge for its verdict in one call and returns it as an `AnswerEval`. `completion` is LiteLLM's function, as
in the lesson's `evaluation/eval.py`. Make one call with the model `model`, the messages from
`judge_messages(question, answer, reference)`, and `AnswerEval` itself as the response format. With a pydantic class
as the response format, the reply's text is JSON that fits it: read it back into an `AnswerEval` and return that.

<details><summary>Hint</summary>

```python
response = completion(model=model, messages=judge_messages(question, answer, reference), response_format=AnswerEval)
return AnswerEval.model_validate_json(response.choices[0].message.content)
```

</details>

In [ ]:
def judge(completion, question, answer, reference, model="gpt-4.1-nano"):
    # Write judge here, as described above.
    raise NotImplementedError("judge is not written yet")

In [ ]:
check("judge")

### 4 · `reciprocal_rank(docs, source)`

Scores one retrieval: 1 divided by the rank of the first document that comes from the expected file. `docs` are the
retrieved LangChain documents, most similar first. `source` is the expected file's path with forward slashes, as your
plan's questions write it, such as `plan/docs/events/royal-rumble-1988.md`.

A document comes from that file when the last two parts of its `"source"` metadata, the folder and the file name, are
those of `source`. So `../../docs/events/royal-rumble-1988.md` matches, however the path starts. Turn every backslash
into `/` first, as Windows writes paths with them. Count ranks from 1. Return 0.0 when no document comes from the
file, or `docs` is empty.

<details><summary>Hint</summary>

```python
want = source.replace("\\", "/").split("/")[-2:]
for rank, doc in enumerate(docs, start=1):
    if doc.metadata["source"].replace("\\", "/").split("/")[-2:] == want:
        ...
```

</details>

In [ ]:
def reciprocal_rank(docs, source):
    # Write reciprocal_rank here, as described above.
    raise NotImplementedError("reciprocal_rank is not written yet")

In [ ]:
check("reciprocal_rank")

### 5 · `ndcg(docs, source, k=10)`

Scores the whole ranking of one retrieval, as the lesson's `calculate_ndcg` does, rather than only its first hit.
Look at the first `k` documents, and give each a relevance of 1 when it comes from the expected file, by the same test
as `reciprocal_rank`, or 0 when it does not. The DCG adds up each relevance divided by the base-2 logarithm of its
rank plus 1, counting ranks from 1, so a hit counts for less the lower it comes. The ideal DCG is the DCG of the same
relevances sorted with the 1s first. Return the DCG divided by the ideal DCG, or 0.0 when nothing is relevant.

<details><summary>Hint</summary>

```python
relevances = [1 if ... else 0 for doc in docs[:k]]
dcg = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(relevances, start=1))
ideal = ...  # the same sum over sorted(relevances, reverse=True)
```

</details>

In [ ]:
def ndcg(docs, source, k=10):
    # Write ndcg here, as described above.
    raise NotImplementedError("ndcg is not written yet")

In [ ]:
check("ndcg")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 5, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Each judgment from `gpt-4.1-nano` costs a fraction of a cent. This reopens the `vector_db` store that Day 2's Try it for real built at the top of your fork.

1. Load your keys with `load_dotenv(override=True)`. Then rebuild Day 3's pipeline: the same embeddings, the Chroma
   store in `ROOT / "vector_db"` (the setup cell set `ROOT` to your fork's folder) with its retriever, and
   `ChatOpenAI(temperature=0, model_name="gpt-4.1-nano")`.
2. Write three test cases of your own, each a question, a reference answer and the file that holds the answer, as
   your plan's questions do (the lesson's `tests.jsonl` lists keywords instead). Take them from Block 4 if you have
   written it.
3. For each case: retrieve, and print `reciprocal_rank(docs, source)` and `ndcg(docs, source)`; answer from the
   retrieved documents, as your Day 3 `answer_question` does; then judge the answer with
   `judge(completion, question, answer, reference)`, using `completion` from `litellm`, and print the scores and the
   feedback.
4. Print the MRR, which is the mean of the reciprocal ranks, the mean nDCG, and the mean accuracy. Optional: run the
   lesson's own dashboard on Insurellm, from the `week5` folder: `uv run implementation/ingest.py`, then
   `uv run evaluator.py`. It costs a few cents: embedding the Insurellm documents with `text-embedding-3-large`, and
   about 300 `gpt-4.1-nano` calls to answer and judge the lesson's 150 questions.

<details><summary>Stuck? One way to write it</summary>

```python
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from litellm import completion

load_dotenv(override=True)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
retriever = Chroma(persist_directory=str(ROOT / "vector_db"), embedding_function=embeddings).as_retriever()
llm = ChatOpenAI(temperature=0, model_name="gpt-4.1-nano")

CASES = [  # (question, reference answer, the file that holds it): write three of your own
    ("Who won the first Royal Rumble match?", "Hacksaw Jim Duggan won it, in 1988.",
     "plan/docs/events/royal-rumble-1988.md"),
]
ranks, gains, accuracy = [], [], []
for question, reference, source in CASES:
    docs = retriever.invoke(question)
    ranks.append(reciprocal_rank(docs, source))
    gains.append(ndcg(docs, source))
    context = "\n\n".join(doc.page_content for doc in docs)
    system = f"You answer questions about WWE history from this context. If it does not say, say so.\n\n{context}"
    answer = llm.invoke([SystemMessage(content=system), HumanMessage(content=question)]).content
    verdict = judge(completion, question, answer, reference)
    accuracy.append(verdict.accuracy)
    print(f"{question}\n  rank {ranks[-1]:.2f} | nDCG {gains[-1]:.2f} | accuracy {verdict.accuracy} | {verdict.feedback}")
print(f"MRR {sum(ranks) / len(ranks):.2f}, mean nDCG {sum(gains) / len(gains):.2f}, "
      f"mean accuracy {sum(accuracy) / len(accuracy):.2f}")
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.